# 01 — MetaForm GYM data overview

Objective: inspect privacy-preserving fitness datasets before any modeling. Available entities are body measurements, completed workout activity, per-source strength, stored nutrition calculations, and minimal fitness context. Authentication, names, notes, and raw UUIDs are excluded. Results are descriptive and may be limited by short or sparse histories.

In [ ]:
import pandas as pd
from metaform_ds.quality import build_quality_report
from metaform_ds.utils.notebooks import load_notebook_data
data = load_notebook_data()
print(f'DATA MODE: {data.mode}')
if data.mode == 'SYNTHETIC': print('SYNTHETIC demonstration only — no PostgreSQL rows were read or written.')

In [ ]:
datasets = {'body_weight': (data.body_weight, 'measured_at'), 'strength': (data.strength, 'session_date'), 'workouts': (data.workouts, 'session_date'), 'nutrition': (data.nutrition, 'calculated_at'), 'fitness_context': (data.fitness_context, None)}
overview = []
for name, (frame, time_col) in datasets.items():
    dates = pd.to_datetime(frame[time_col], utc=True, errors='coerce') if time_col and time_col in frame else pd.Series(dtype='datetime64[ns, UTC]')
    overview.append({'dataset': name, 'rows': len(frame), 'columns': len(frame.columns), 'date_min': dates.min() if len(dates) else pd.NaT, 'date_max': dates.max() if len(dates) else pd.NaT, 'missing_cells': int(frame.isna().sum().sum())})
pd.DataFrame(overview)

In [ ]:
for name, (frame, time_col) in datasets.items():
    print(f'\n{name}: {list(frame.columns)}')
    report = build_quality_report(frame, dataset_name=name, timestamp_column=time_col)
    print(report.null_rates.to_string(index=False))
    print(report.issues.to_string(index=False) if not report.issues.empty else 'No configured issues detected.')
print('\nLimitations: observational data, UTC calendar semantics, nullable legacy nutrition context, duplicate measurement timestamps retained, and no inference when histories are sparse.')